# Transform: Cleaning the Data — Practitioner Walkthrough
**Module 3 · outpatient_visits.csv · Habeeba Siddiqui**

---

This notebook is the practitioner's version of the Transform step. It shows the decisions, the code, and the reasoning at each stage. Compare it against your own notebook after you have completed yours.

Two rules that applied throughout:

1. **The brief is the scope.** Fix what would break the analysis. Leave everything else alone.
2. **Never overwrite the raw file.** All changes are made to a working copy.

---

### The brief

> Who uses the clinic and how often, centred on repeat visitors and uninsured patients, built for the board and reusable for the county funder report.

**Repeat visitor:** a unique patient who returned within one month of their preceding visit.  
**Uninsured:** patients with no coverage at all.  
**Unit of analysis:** patients, not visits.

---
## Section 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

url = 'https://raw.githubusercontent.com/EDC-AICC/ai-ds-development/main/src/data/outpatient_visits.csv'

# Load the raw file — this dataframe is never modified
raw = pd.read_csv(url)

# All cleaning happens on this working copy
df = raw.copy()

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

Shape: (812, 13)
Columns: ['visit_id', 'patient_id', 'visit_date', 'icd_code', 'icd_description', 'visit_type', 'gender', 'age', 'insurance_type', 'provider_id', 'county', 'copay_amount', 'follow_up_required']


In [2]:
# Quick profile of the columns that will need cleaning
# Run this before writing any cleaning prompt — it shows nulls, range,
# top categories, and unique counts all in one place.
# cols = ['visit_date', 'insurance_type', 'visit_type', 'age', 'copay_amount', 'county']
# df[cols].describe(include='all')
df.describe(include='all')

,visit_id,patient_id,visit_date,icd_code,icd_description,visit_type,gender,age,insurance_type,provider_id,county,copay_amount,follow_up_required
count,812,812,812,783,783,812,812,779.000000,812,812,564,768.000000,812
unique,797,200,676,16,16,16,14,NaN,18,8,8,NaN,10
top,V0143,P034,10-08-2022,I10,Essential Hypertension,office visit,female,NaN,mcd,DR08,Los Angeles,NaN,no
freq,2,8,3,98,98,92,119,NaN,88,110,86,NaN,107
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.743261,NaN,NaN,NaN,35.384115,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.153209,NaN,NaN,NaN,47.679335,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-5.000000,NaN,NaN,NaN,-127.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.000000,NaN,NaN,NaN,0.000000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.000000,NaN,NaN,NaN,24.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.500000,NaN,NaN,NaN,44.250000,NaN


In [3]:
# All unique values in categorical columns — describe() only shows the top one
cat_cols = ['insurance_type', 'visit_type']
for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts(dropna=False).to_string())


--- insurance_type ---
insurance_type
mcd           88
medicaid      82
MEDICAID      80
Medicaid      56
MEDICARE      55
Private       55
Commercial    51
Medicare      51
PRIVATE       46
private       45
self-pay      38
UNINSURED     38
MCR           36
medicare      33
uninsured     18
commercial    15
Self-Pay      15
Uninsured     10

--- visit_type ---
visit_type
office visit    92
OFFICE VISIT    88
Office Visit    79
Office visit    78
Telehealth      62
telehealth      58
Follow-Up       45
follow-up       42
followup        40
TELEHEALTH      40
Tele-health     39
FOLLOW UP       39
Follow Up       34
Urgent Care     28
URGENT CARE     28
urgent care     20


In [4]:
# Summary stats for numeric columns
# Use this to spot out-of-range values before writing your cleaning prompt
import numpy as np
num_cols = ['age', 'copay_amount']
stats = df[num_cols].agg(['min', 'max', 'mean', 'median']).round(2)
print(stats.to_string())
print()
for col in num_cols:
    print(f'{col} nulls: {df[col].isna().sum()}')

           age  copay_amount
min      -5.00       -127.00
max     999.00        200.00
mean     54.74         35.38
median   51.00         24.00

age nulls: 33
copay_amount nulls: 44


---
## Section 1 — visit_date: Parse Date Column

**Finding from Explore:** `visit_date` is stored as object (string). It has not been parsed as a date.

**Why it matters:** The repeat visitor calculation needs date arithmetic — specifically, the gap in days between a patient's consecutive visits. That is impossible on a string.

---

**Practitioner decision:**

> First pass: parse in place using `pd.to_datetime` with `errors='coerce'`. This handles the standard ISO format. Any value that fails becomes NaT — but before accepting those as truly unparseable, I inspect the original strings. If they contain digits, they may be real dates in a format pandas did not recognise on the first pass. In this dataset, the failed rows turned out to be US-format dates with mixed separators (`MM-DD-YYYY` and `MM/DD/YYYY`). Second pass: re-parse only those rows using `format='mixed'` with `dayfirst=False`. I do not drop rows at any stage. I verify after each pass.

---
### Prompt that generated this code

```
My visit_date column is stored as a string with mixed formats.
Write a two-pass parse:
  Pass 1 — pd.to_datetime(errors='coerce') on the whole column.
  Inspect — identify which original strings became NaT; check if they
             contain digits (real dates in an unrecognised format).
  Pass 2 — re-parse digit-containing failures with format='mixed',
             dayfirst=False.
Report the NaT count after each pass separately.
Do not drop any rows at any stage.
```

**Why the prompt is written this way**
Asking for a two-pass approach with an inspect step in between forces AI to treat
NaTs as a question, not a final answer. Without that instruction, AI stops at
Pass 1 and accepts every unrecognised string as a genuine null — losing real dates.
Specifying `dayfirst=False` prevents AI from swapping day and month on US-format dates.

**Watch for in AI output**
- Does it include the inspect step, or jump straight from Pass 1 to Pass 2?
- Is `dayfirst=False` set? If omitted, `01-08-2022` parses as 1 August not 8 January.
- Does it report NaT counts after *each* pass separately — not just the final total?
- Does it avoid dropping rows at any stage?

In [5]:
# PASS 1 — Standard parse
# Store the null mask from raw so we can identify rows that fail the parse
was_null_before = raw['visit_date'].isna()
nulls_before = was_null_before.sum()
print(f'Nulls before parsing: {nulls_before}')

# Parse to datetime — coerce unparseable values to NaT
df['visit_date'] = pd.to_datetime(df['visit_date'], errors='coerce')

nulls_after = df['visit_date'].isna().sum()
print(f'Nulls after first pass: {nulls_after}')
print(f'New NaTs created: {nulls_after - nulls_before}')

Nulls before parsing: 0
Nulls after first pass: 545
New NaTs created: 545


In [6]:
# INSPECT — What are the strings that became NaT?
# We look at raw (not df) because df already has NaT in those positions
is_null_after    = df['visit_date'].isna()
newly_nulled_idx = df[~was_null_before & is_null_after].index

print(f'Rows that became NaT during parsing: {len(newly_nulled_idx)}')
print()

original_strings = raw.loc[newly_nulled_idx, 'visit_date']

print('Original string values:')
print(original_strings.value_counts(dropna=False))
print()

# Check for digits — digits mean a real date in a format pandas did not recognise
has_digits = original_strings.str.contains(r'\d', na=False)
print(f'Values containing digits (potentially salvageable): {has_digits.sum()}')

Rows that became NaT during parsing: 545

Original string values:
visit_date
08-05-2023    3
10-08-2022    3
03/03/2023    3
01-14-2022    3
07/15/2023    3
             ..
01/07/2023    1
06/09/2022    1
12-25-2023    1
05/31/2023    1
11-02-2022    1
Name: count, Length: 455, dtype: int64

Values containing digits (potentially salvageable): 544


**What the inspect output tells you:**
- Values with digits are real dates in a format pandas did not recognise — salvageable with a second parse.
- Empty strings, whitespace, or text like `unknown` are genuinely null — leave as NaT.

In this dataset, the failed strings are US-format dates: `MM-DD-YYYY` and `MM/DD/YYYY`. Pandas defaulted to ISO format on the first pass and could not read them. The second pass below fixes them.

In [7]:
# PASS 2 — Re-parse the failed rows with mixed US format
# Pull original strings from raw — df already has NaT in those positions
reparsed = pd.to_datetime(
    raw.loc[newly_nulled_idx, 'visit_date'],
    format='mixed',
    dayfirst=False,   # month first — US convention
    errors='coerce'
)

# Fill only the NaT positions — rows that parsed correctly on pass 1 are not touched
df.loc[newly_nulled_idx, 'visit_date'] = reparsed

still_nat = df.loc[newly_nulled_idx, 'visit_date'].isna().sum()
print(f'Rows fixed by second pass: {len(newly_nulled_idx) - still_nat}')
print(f'Still NaT after second pass: {still_nat}')
print(f'Total NaTs in visit_date now: {df["visit_date"].isna().sum()}')

Rows fixed by second pass: 541
Still NaT after second pass: 4
Total NaTs in visit_date now: 4


In [8]:
# If anything still NaT, inspect those strings before accepting them as null
remaining_idx = df[(~was_null_before) & (df['visit_date'].isna())].index

if len(remaining_idx) > 0:
    print('Strings that could not be parsed after both passes:')
    print(raw.loc[remaining_idx, 'visit_date'].value_counts(dropna=False))
    print()
    print('Full rows:')
    print(raw.loc[remaining_idx])
else:
    print('All non-null date strings parsed successfully across both passes.')

Strings that could not be parsed after both passes:
visit_date
2023-00-00      1
not recorded    1
13/45/2023      1
Feb 30 2023     1
Name: count, dtype: int64

Full rows:
    visit_id patient_id    visit_date icd_code             icd_description  \
172    V0273       P065    2023-00-00    M54.5               Low Back Pain   
377    V0167       P039  not recorded    E78.5              Hyperlipidemia   
387    V0377       P094    13/45/2023    M54.5               Low Back Pain   
676    V0422       P103   Feb 30 2023      Z23  Encounter for Immunization   

      visit_type  gender   age insurance_type provider_id         county  \
172  urgent care  FEMALE  80.0       Medicaid        DR07         Fresno   
377   telehealth       f  25.0       medicaid        DR05         Fresno   
387    follow-up  female  34.0            mcd        DR06            NaN   
676   Telehealth       F  53.0        Private        DR07  San Francisco   

     copay_amount follow_up_required  
172           0.

In [9]:
# Final verification
print('dtype:', df['visit_date'].dtype)
print('Date range:', df['visit_date'].min(), 'to', df['visit_date'].max())
print('Total NaTs remaining:', df['visit_date'].isna().sum())

dtype: datetime64[ns]
Date range: 2022-01-02 00:00:00 to 2023-12-31 00:00:00
Total NaTs remaining: 4


**What to look for:**  
dtype should now be `datetime64[ns]`. Date range should span the two years in the dataset. After both passes, the only remaining NaTs should be values that were genuinely unreadable — not real dates in a different format.

**The principle this illustrates:** A NaT after the first parse is not automatically a null. Always inspect the original string before accepting the loss. In this dataset, one extra step recovered all the failed rows.

**Log entry:**

| Field | Entry |
|-------|-------|
| Column | visit_date |
| Problem | Stored as object; mixed date formats (ISO, MM-DD-YYYY, MM/DD/YYYY) |
| Decision | Two-pass parse: standard first, inspect failures, then mixed US format for rows with digits; do not drop rows |
| Action | Pass 1: `pd.to_datetime(errors='coerce')` → Pass 2: `pd.to_datetime(format='mixed', dayfirst=False)` on failed rows |
| Rows affected | 0 dropped; all digit-containing strings recovered |
| Verified? | Yes — dtype, date range, NaT count after each pass |

---
## Section 2 — insurance_type: Standardise Values

**Finding from Explore:** `insurance_type` may contain values that differ only by case or whitespace.

**Why it matters:** The uninsured patient count depends entirely on this column grouping correctly. `Uninsured` and `uninsured` as separate entries would split the group and undercount it.

---

**Practitioner decision:**

> Strip whitespace first, then apply title case. Title case is the right choice here because the values will appear in board-level and funder-level output — they need to read as proper labels, not all-caps or all-lowercase. I check distinct values before and after so the comparison is visible and auditable.

---
### Prompt that generated this code

```
Show me value_counts(dropna=False) for insurance_type so I can see all variants.
Then standardise the column:
  1. Strip whitespace.
  2. Apply title case.
  3. Apply this replacement map for anything title case cannot fix:
       'Mcd' → 'Medicaid'
       'Mcr' → 'Medicare'
Show value_counts() before and after. Do not drop rows or create new columns.
```

**Why the prompt is written this way**
The replacement map contains domain-specific abbreviations — AI cannot guess
that `mcd` means Medicaid without being told. Looking at value_counts() *before*
writing this prompt is what surfaces those abbreviations so you can include them.
Step order matters: strip → title case → replace. Reversing title case and replace
means the map keys won't match.

**Watch for in AI output**
- Does the after value_counts show exactly the expected categories and nothing else?
- Any unexpected values still showing? A variant was missed — revise the map and re-run.
- Total row count unchanged — the same rows, just redistributed across categories.
- Is the replacement map applied *after* title case, not before?

In [10]:
# Distinct values BEFORE — so we can see exactly what changed
print('insurance_type values BEFORE:')
print(df['insurance_type'].value_counts(dropna=False))
print()

insurance_type values BEFORE:
insurance_type
mcd           88
medicaid      82
MEDICAID      80
Medicaid      56
MEDICARE      55
Private       55
Commercial    51
Medicare      51
PRIVATE       46
private       45
self-pay      38
UNINSURED     38
MCR           36
medicare      33
uninsured     18
commercial    15
Self-Pay      15
Uninsured     10
Name: count, dtype: int64



In [11]:
# Step 1 — Strip whitespace
df['insurance_type'] = df['insurance_type'].str.strip()

# Step 2 — Title case (fixes all the MEDICAID / medicaid / Medicaid variants)
df['insurance_type'] = df['insurance_type'].str.title()

# Step 3 — Replace abbreviations that title case cannot fix
abbreviation_map = {
    'Mcd': 'Medicaid',
    'Mcr': 'Medicare',
}
df['insurance_type'] = df['insurance_type'].replace(abbreviation_map)

print('insurance_type values AFTER:')
print(df['insurance_type'].value_counts(dropna=False))

insurance_type values AFTER:
insurance_type
Medicaid      306
Medicare      175
Private       146
Uninsured      66
Commercial     66
Self-Pay       53
Name: count, dtype: int64


**What to look for:**  
The number of distinct values should be the same as or fewer than before. If it is fewer, values that were duplicated by case or whitespace have been merged — that is the fix working. The counts for each group may have increased if merging happened.

**Log entry:**

| Field | Entry |
|-------|-------|
| Column | insurance_type |
| Problem | Possible case/whitespace inconsistencies |
| Decision | Strip whitespace; apply title case |
| Action | `.str.strip().str.title()` |
| Rows affected | All non-null rows |
| Verified? | Yes — distinct values before and after |

---
## Section 3 — visit_type: Standardise Values

**Finding from Explore:** `visit_type` may contain the same case/whitespace inconsistencies as `insurance_type`.

**Why it matters:** Two separate brief requirements depend on this column. First, the care coordinator hiring decision rests partly on what repeat visitors are coming back for — if they cycle back for the same visit type, that tells the director something specific about clinical need. Second, the county funder will want to see utilisation by visit type. A fragmented column produces a fragmented view of both.

---

**Practitioner decision:**

> Same approach as `insurance_type` — strip then title case, for the same reason. Consistency across categorical columns also matters: if `insurance_type` is title case and `visit_type` is lowercase, any cross-tabulation or grouped output will look inconsistent.

---
### Prompt that generated this code

```
Standardise the visit_type column in df to exactly four values:
'Office Visit', 'Telehealth', 'Follow-Up', 'Urgent Care'.
  1. Strip whitespace.
  2. Apply title case.
  3. Apply this replacement map for residuals:
       'Tele-Health' → 'Telehealth'
       'Follow Up'   → 'Follow-Up'
       'Followup'    → 'Follow-Up'
Show value_counts() before and after.
```

**Why the prompt is written this way**
Title case creates predictable edge cases with hyphens and merged words.
`tele-health` becomes `Tele-Health` under title case — not `Telehealth`.
`followup` becomes `Followup` — not `Follow-Up`. Without an explicit
replacement map for these residuals, they become nulls silently.
Stating the exact four target values gives AI a clear success criterion.

**Watch for in AI output**
- Exactly four values in after value_counts — no more, no less.
- Hyphen variants (`Tele-Health`) and merged words (`Followup`) explicitly 
  handled in the map.
- Replacement map applied *after* title case — if applied before, the keys 
  won't match the title-cased strings.

In [12]:
# Distinct values BEFORE
print('visit_type values BEFORE:')
print(df['visit_type'].value_counts(dropna=False))
print()

visit_type values BEFORE:
visit_type
office visit    92
OFFICE VISIT    88
Office Visit    79
Office visit    78
Telehealth      62
telehealth      58
Follow-Up       45
follow-up       42
followup        40
TELEHEALTH      40
Tele-health     39
FOLLOW UP       39
Follow Up       34
Urgent Care     28
URGENT CARE     28
urgent care     20
Name: count, dtype: int64



In [13]:
# Step 1 — Strip whitespace
df['visit_type'] = df['visit_type'].str.strip()

# Step 2 — Title case (handles Office Visit, Urgent Care, and most others)
df['visit_type'] = df['visit_type'].str.title()

# Step 3 — Fix what title case cannot: hyphen variants and merged words
visit_type_map = {
    'Tele-Health': 'Telehealth',  # hyphen caused capitalisation of 'health'
    'Follow Up':   'Follow-Up',   # FOLLOW UP and Follow Up both land here
    'Followup':    'Follow-Up',   # no separator — title case gives one word
}
df['visit_type'] = df['visit_type'].replace(visit_type_map)

print('visit_type values AFTER:')
print(df['visit_type'].value_counts(dropna=False))

visit_type values AFTER:
visit_type
Office Visit    337
Follow-Up       200
Telehealth      199
Urgent Care      76
Name: count, dtype: int64


**Log entry:**

| Field | Entry |
|-------|-------|
| Column | visit_type |
| Problem | Possible case/whitespace inconsistencies |
| Decision | Strip whitespace; apply title case — consistent with insurance_type |
| Action | `.str.strip().str.title()` |
| Rows affected | All non-null rows |
| Verified? | Yes — distinct values before and after |

---
## Section 4 — county: Document and Flag Missing Values

**Finding from Explore:** `county` has 248 missing values — 30% of the dataset.

**Why it matters:** The county funder needs a geographic cut. If 30% of county values are missing, that cut is incomplete. The more important question is whether the missing values are concentrated in the groups the brief cares about most.

**The constraint:** I cannot impute or invent county. I have no secondary source. Dropping rows is not an option — I would lose 30% of the data and introduce bias into every metric.

---

**Practitioner decision:**

> First, understand the pattern of missingness — is it concentrated in uninsured patients or a particular visit type? That shapes how I describe the limitation. Then add a `county_known` flag column so any analysis that uses county can filter explicitly rather than silently dropping nulls. Leave the nulls in `county` as null. Document the limitation so it travels with every deliverable that uses this column.

---
### Prompt that generated this code

```
The county column has 248 missing values — 30% of rows.
I cannot impute county from other columns and I cannot drop rows.
Do two things:
  1. Add a boolean column county_known (True where county is not null,
     False where it is). Leave the county column itself unchanged.
  2. Show a breakdown of missing county by insurance_type and by visit_type
     so I can see if missingness is concentrated in specific groups.
```

**Why the prompt is written this way**
Stating the constraints upfront — can't impute, can't drop — prevents AI
from defaulting to those approaches, which is what it will suggest without
guidance. Asking for the group breakdown turns a data quality problem into
a finding: if missingness is concentrated in one group, that shapes how
you describe the limitation in any deliverable that uses county.

**Watch for in AI output**
- County null count must be unchanged after the flag is added — AI sometimes
  fills nulls as a side effect of adding the flag.
- `county_known` should be True/False (boolean), not 1/0 or inverted.
- Does it produce the by-group breakdown? That's what tells you whether 
  the missingness is random or structural.

In [14]:
# Understand the pattern of missingness before deciding
missing_county = df[df['county'].isna()]

print(f'Total rows missing county: {len(missing_county)} ({len(missing_county)/len(df)*100:.1f}%)')
print()

print('Missing county by insurance_type:')
print(missing_county['insurance_type'].value_counts(dropna=False))
print()

print('Missing county by visit_type:')
print(missing_county['visit_type'].value_counts(dropna=False))

Total rows missing county: 248 (30.5%)

Missing county by insurance_type:
insurance_type
Medicaid      111
Medicare       46
Private        41
Commercial     31
Self-Pay       11
Uninsured       8
Name: count, dtype: int64

Missing county by visit_type:
visit_type
Office Visit    101
Follow-Up        72
Telehealth       56
Urgent Care      19
Name: count, dtype: int64


In [15]:
# Add a flag column — True where county is known, False where missing
df['county_known'] = df['county'].notna()

print('county_known value counts:')
print(df['county_known'].value_counts())

county_known value counts:
county_known
True     564
False    248
Name: count, dtype: int64


In [16]:
# Verification — nulls in county should be unchanged; flag column should exist
print('County nulls remaining:', df['county'].isna().sum())
print('New columns added:', [c for c in df.columns if c not in raw.columns])

County nulls remaining: 248
New columns added: ['county_known']


**What to look for:**  
If missing county is concentrated in uninsured patients, note it explicitly — the county funder's most important group is also the least geographically complete. That is a finding, not just a data quality note, and it belongs in the deliverable.

**Limitation statement for any deliverable using county:**  
*County data is available for 70% of records. Geographic breakdowns in this report reflect only those visits where county was recorded. If the pattern of missing county is not random across patient groups, county-level figures may not be representative of the full patient population.*

**Log entry:**

| Field | Entry |
|-------|-------|
| Column | county |
| Problem | 248 missing values (30%) |
| Decision | Leave nulls; add county_known flag; document limitation |
| Ruled out | Dropping rows (loses 30% of data); imputing (no secondary source) |
| Action | `df['county_known'] = df['county'].notna()` |
| Rows affected | 0 dropped; 1 column added |
| Limitation | County breakdowns cover 70% of records; concentration of missing values by group to be noted in deliverable |
| Verified? | Yes — null count unchanged; flag column present |

---
## Section 5 — age and copay_amount: Out-of-Range Values

**Finding from Explore:** `age` has 33 nulls and possible values outside 0–110. `copay_amount` has 44 nulls and possible negative values.

**Why it matters:** Neither column is a primary metric in the brief. But both may appear in subgroup breakdowns, and a value of age = -3 or copay = -500 in a summary table would raise questions about data quality that undermine the whole deliverable.

---

**Practitioner decision:**

> Set implausible values to null — do not drop the row. The rest of the record is valid. Age threshold: below 0 or above 110. Copay threshold: below 0. Existing nulls stay as null — I have no basis for imputing age or copay, and imputing would introduce assumptions the brief does not support. Two separate steps, one column at a time — not chained.

---
### Prompt that generated this code

```
In df, set out-of-range values to NaN. Do not drop rows — only null the cell.
Handle each column separately:
  - age: values below 0 or above 110 → NaN
  - copay_amount: negative values → NaN
Print min, max, and null count for each column before and after.
Do not chain these operations.
```

**Why the prompt is written this way**
"Do not drop rows" must be stated explicitly — AI defaults to removing the row.
The rest of the record is valid; only the implausible cell should change.
Thresholds (0, 110, negative) are domain decisions — AI would use a generic
default like 0–120 for age without being told the right range for this population.
Handling columns separately with separate output makes each change independently
verifiable instead of mixing results together.

**Watch for in AI output**
- Does it null the cell or drop the row? Check with `df.loc[age_invalid, 'age'] = np.nan`
  pattern — not `df = df[~mask]` which drops the row.
- Null count after = null count before + number of out-of-range values found.
  If it's higher, something else was converted unexpectedly.
- Separate before/after output for each column — not combined.

In [17]:
# age — check range before
print('age BEFORE:')
print(f'  min: {df["age"].min()}, max: {df["age"].max()}, nulls: {df["age"].isna().sum()}')

# Identify out-of-range values
age_too_low  = df['age'] < 0
age_too_high = df['age'] > 110
age_invalid  = age_too_low | age_too_high
print(f'  Out-of-range values: {age_invalid.sum()}')

# Set out-of-range to null
df.loc[age_invalid, 'age'] = np.nan

print('age AFTER:')
print(f'  min: {df["age"].min()}, max: {df["age"].max()}, nulls: {df["age"].isna().sum()}')

age BEFORE:
  min: -5.0, max: 999.0, nulls: 33
  Out-of-range values: 10
age AFTER:
  min: 18.0, max: 85.0, nulls: 43


In [18]:
# copay_amount — check range before
print('copay_amount BEFORE:')
print(f'  min: {df["copay_amount"].min()}, max: {df["copay_amount"].max()}, nulls: {df["copay_amount"].isna().sum()}')

# Identify negative values
copay_negative = df['copay_amount'] < 0
print(f'  Negative values: {copay_negative.sum()}')

# Set negatives to null
df.loc[copay_negative, 'copay_amount'] = np.nan

print('copay_amount AFTER:')
print(f'  min: {df["copay_amount"].min()}, max: {df["copay_amount"].max()}, nulls: {df["copay_amount"].isna().sum()}')

copay_amount BEFORE:
  min: -127.0, max: 200.0, nulls: 44
  Negative values: 14
copay_amount AFTER:
  min: 0.0, max: 200.0, nulls: 58


**What to look for:**  
After cleaning, age min should be ≥ 0, max ≤ 110. Copay min should be ≥ 0. The null count for each column may be slightly higher than before — that is expected, because implausible values have been converted to null rather than dropped.

**Log entry:**

| Field | Entry |
|-------|-------|
| Columns | age, copay_amount |
| Problem | Out-of-range values; existing nulls |
| Decision (age) | Set values below 0 or above 110 to null; leave existing nulls |
| Decision (copay) | Set negative values to null; leave existing nulls |
| Action | `df.loc[age_invalid, 'age'] = np.nan`; `df.loc[copay_negative, 'copay_amount'] = np.nan` |
| Rows affected | Only the cells with implausible values; no rows dropped |
| Verified? | Yes — min/max/null count before and after |

---
## Section 6 — Duplicates

**Finding from Explore:** `visit_id` and `patient_id` + `visit_date` were flagged as unchecked for duplicates.

**Why it matters:** A duplicate visit row inflates visit counts. In the repeat visitor calculation, a patient appearing twice on the same date would look like two visits — potentially qualifying them as a repeat visitor when they are not.

---

**Practitioner decision:**

> Check first — see what is actually there before deciding anything. If `visit_id` appears more than once and all columns are identical, it is a true duplicate — a row that was written twice. Keep the first occurrence, drop the second. If `visit_id` appears more than once with different column values, that is a data integrity problem that cannot be resolved by dropping — it needs to be flagged and escalated. I do not silently drop a row when I do not know which version is correct.

---
### Prompt that generated this code

```
Check df for duplicate records in two ways:
  1. Rows sharing the same visit_id — are they fully identical across all
     columns, or do any conflict (same ID, different values)?
  2. Rows sharing the same patient_id + visit_date combination.
Show counts and sample rows for each check. Do not remove anything yet.
Then, only if all visit_id duplicates are fully identical, drop them with
keep='first' and report how many rows were removed.
```

**Why the prompt is written this way**
The default AI response to "remove duplicates" is `drop_duplicates()` applied
immediately — no check, no diagnostic. A conflicting record (same visit_id,
different values) would have one version silently deleted with no warning.
Separating the check from the drop, and conditioning the drop on fully identical
rows, forces AI to include the diagnostic step. "Do not remove anything yet"
is the key phrase that prevents premature action.

**Watch for in AI output**
- Is the check step separate from the drop step?
- Does it report conflicting visit_ids explicitly — even if the count is zero?
- Row count after drop = rows before − number of fully identical duplicate rows.
- If conflicting records exist: they should be flagged, not dropped.

In [19]:
# Check visit_id duplicates
visit_id_dup_mask = df.duplicated(subset='visit_id', keep=False)
visit_id_dups = df[visit_id_dup_mask]
print(f'Rows with a duplicate visit_id: {len(visit_id_dups)}')

if len(visit_id_dups) > 0:
    # Check whether the duplicated rows are fully identical
    fully_identical = df.duplicated(keep=False)
    print(f'Of those, fully identical rows: {fully_identical.sum()}')
    print()
    print('Duplicate visit_id rows (first 10):')
    print(visit_id_dups.sort_values('visit_id').head(10))

Rows with a duplicate visit_id: 30
Of those, fully identical rows: 30

Duplicate visit_id rows (first 10):
    visit_id patient_id visit_date icd_code  \
293    V0031       P008 2023-08-05    E78.5   
300    V0031       P008 2023-08-05    E78.5   
497    V0044       P010 2023-06-16      I10   
110    V0044       P010 2023-06-16      I10   
41     V0060       P013 2023-01-11    J06.9   
460    V0060       P013 2023-01-11    J06.9   
104    V0143       P033 2023-06-27      NaN   
617    V0143       P033 2023-06-27      NaN   
59     V0210       P050 2023-02-22    J06.9   
647    V0210       P050 2023-02-22    J06.9   

                       icd_description    visit_type gender   age  \
293                     Hyperlipidemia  Office Visit   male  25.0   
300                     Hyperlipidemia  Office Visit   male  25.0   
497             Essential Hypertension  Office Visit      f  45.0   
110             Essential Hypertension  Office Visit      f  45.0   
41   Acute Upper Respiratory I

In [20]:
# Check patient_id + visit_date duplicates
pt_date_dup_mask = df.duplicated(subset=['patient_id', 'visit_date'], keep=False)
pt_date_dups = df[pt_date_dup_mask]
print(f'Rows with duplicate patient_id + visit_date: {len(pt_date_dups)}')

if len(pt_date_dups) > 0:
    print()
    print('Sample (first 10):')
    print(pt_date_dups.sort_values(['patient_id', 'visit_date']).head(10))

Rows with duplicate patient_id + visit_date: 34

Sample (first 10):
    visit_id patient_id visit_date icd_code  \
293    V0031       P008 2023-08-05    E78.5   
300    V0031       P008 2023-08-05    E78.5   
110    V0044       P010 2023-06-16      I10   
497    V0044       P010 2023-06-16      I10   
41     V0060       P013 2023-01-11    J06.9   
460    V0060       P013 2023-01-11    J06.9   
104    V0143       P033 2023-06-27      NaN   
617    V0143       P033 2023-06-27      NaN   
59     V0210       P050 2023-02-22    J06.9   
647    V0210       P050 2023-02-22    J06.9   

                       icd_description    visit_type gender   age  \
293                     Hyperlipidemia  Office Visit   male  25.0   
300                     Hyperlipidemia  Office Visit   male  25.0   
110             Essential Hypertension  Office Visit      f  45.0   
497             Essential Hypertension  Office Visit      f  45.0   
41   Acute Upper Respiratory Infection  Office Visit      F  73.0   


In [21]:
# Remove fully identical duplicate rows — keep first occurrence
rows_before = len(df)
df = df.drop_duplicates(keep='first')
rows_after = len(df)

print(f'Rows before: {rows_before}')
print(f'Rows after:  {rows_after}')
print(f'Rows removed: {rows_before - rows_after}')
print()
print('visit_id duplicates remaining:', df.duplicated(subset='visit_id').sum())

Rows before: 812
Rows after:  797
Rows removed: 15

visit_id duplicates remaining: 0


**What to look for:**  
If visit_id duplicates remain after dropping fully identical rows, those are conflicting records — same ID, different data. Do not drop them silently. Flag them, note the count, and bring them to the director. They represent a data integrity issue upstream of this dataset.

**Log entry:**

| Field | Entry |
|-------|-------|
| Column(s) | visit_id; patient_id + visit_date |
| Problem | Possible duplicate records |
| What was found | *fill in after running — how many, and of what type* |
| Decision | Drop fully identical rows (keep first); flag conflicting visit_ids for escalation |
| Action | `df.drop_duplicates(keep='first')` |
| Rows removed | *fill in after running* |
| Verified? | Yes — row count before/after; visit_id duplicate count after |

---
## Section 7 — Cleaning Log

Every decision made in this notebook, in one place. This table travels with the clean dataset.

| Section | Column | Problem | Decision | Action | Rows affected | Limitation to document |
|---------|--------|---------|----------|--------|---------------|------------------------|
| 1 | visit_date | Stored as string | Parse in place; coerce errors to NaT; do not drop rows | `pd.to_datetime(errors='coerce')` | 0 dropped | None unless NaT count is high |
| 2 | insurance_type | Case/whitespace inconsistencies | Strip then title case | `.str.strip().str.title()` | All non-null rows | None |
| 3 | visit_type | Case/whitespace inconsistencies | Strip then title case (consistent with insurance_type) | `.str.strip().str.title()` | All non-null rows | None |
| 4 | county | 248 nulls (30%) | Flag with county_known; leave nulls; do not drop or impute | `df['county_known'] = df['county'].notna()` | 0 dropped; 1 column added | County breakdowns cover 70% of records; check if missingness is concentrated in key groups |
| 5 | age | Out-of-range values; 33 nulls | Set implausible values (< 0 or > 110) to null; leave existing nulls | `df.loc[age_invalid, 'age'] = np.nan` | Implausible cells only | None |
| 5 | copay_amount | Negative values; 44 nulls | Set negatives to null; leave existing nulls | `df.loc[copay_negative, 'copay_amount'] = np.nan` | Negative cells only | None |
| 6 | visit_id / duplicates | Possible duplicate rows | Drop fully identical rows; flag conflicting IDs | `drop_duplicates(keep='first')` | Fill in after running | Conflicting visit_ids (if any) need upstream investigation |

---
## Section 8 — Save the Clean Dataset

In [22]:
output_path = 'outpatient_visits_clean.csv'
df.to_csv(output_path, index=False)

print('=== Before / After Summary ===')
print(f'Rows:    {len(raw)} → {len(df)}')
print(f'Columns: {raw.shape[1]} → {df.shape[1]}  (county_known added)')
print()

print('Null counts — key columns:')
cols = ['visit_date', 'insurance_type', 'visit_type', 'county', 'age', 'copay_amount']
for col in cols:
    before = raw[col].isna().sum() if col in raw.columns else 'n/a'
    after  = df[col].isna().sum()  if col in df.columns  else 'n/a'
    print(f'  {col}: {before} → {after}')

print()
print(f'Clean file saved to: {output_path}')

=== Before / After Summary ===
Rows:    812 → 797
Columns: 13 → 14  (county_known added)

Null counts — key columns:
  visit_date: 0 → 4
  insurance_type: 0 → 0
  visit_type: 0 → 0
  county: 248 → 243
  age: 33 → 43
  copay_amount: 44 → 58

Clean file saved to: outpatient_visits_clean.csv


---
## What to Compare Against Your Notebook

Put your notebook next to this one. Three things to look at:

**1. Did you fix anything this notebook did not?**  
If yes, ask whether that fix connects to a metric the brief requires. If it does not, it was out of scope — useful to know, but not part of this analysis.

**2. Did this notebook fix something you skipped?**  
The most common one students miss is `visit_type`. It looks like a reference column — not obviously broken — but its value to the brief only becomes clear when you hold the brief in mind while reading the column list. That is the habit: brief first, then columns.

**3. Do your decisions match, or did you make a different call?**  
A different decision is not automatically wrong. The county section in particular has more than one defensible approach. What matters is that the decision is documented, the reasoning holds against the brief, and the limitation travels with the deliverable.

The clean dataset is now ready for Part 4: analysis.